# Typy danych w Pandas

Trzy tematy:
1. **Redukcja pamięci**, float64 → float32/float16, int64 → int8
2. **datetime**, parsowanie dat i `.dt` accessor
3. **pandas StringDtype**, co zyskujemy względem domyślnego `object`

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id":      ["A","B","C","D","E","F","G","H"],
    "purchase_date": ["2021-01-10 08:30","2021-02-15 22:00","2021-03-01 14:15",
                      "2021-03-20 09:00","2021-04-05 17:45","2021-04-10 11:00",
                      "2021-05-01 08:00","2021-06-15 20:30"],
    "price":         [120.0, 50.0, 200.0, 30.0, 80.0, 155.0, 45.0, 300.0],
    "qty":           [1, 2, 1, 3, 1, 1, 2, 1],
    "category":      ["shoes","clothes","shoes","electronics",
                      "clothes","shoes","electronics","clothes"],
    "city":          ["Kraków","Warszawa","Kraków","Gdańsk",
                      "Poznań","Kraków","Warszawa","Gdańsk"],
    "is_bad":        [0, 0, 1, 0, 0, 0, 0, 1],
})

print(df.dtypes)
print()
print(f"Rozmiar w pamięci: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

## 1. Redukcja pamięci

Domyślnie Pandas używa `float64` i `int64`, 8 bajtów na wartość.
W dużych zbiorach to marnotrawstwo gdy wartości mieszczą się w mniejszym typie.

In [ ]:
# Zakres typów zmiennoprzecinkowych
print(f"float64: {np.finfo(np.float64).min:.0e}  do  {np.finfo(np.float64).max:.0e}")
print(f"float32: {np.finfo(np.float32).min:.0e}  do  {np.finfo(np.float32).max:.0e}")
print(f"float16: {np.finfo(np.float16).min:.0e}  do  {np.finfo(np.float16).max:.0e}")
print()
print(f"int64:  {np.iinfo(np.int64).min}  do  {np.iinfo(np.int64).max}")
print(f"int8:   {np.iinfo(np.int8).min}   do  {np.iinfo(np.int8).max}")

In [ ]:
df_opt = df.copy()

df_opt["price"]  = df_opt["price"].astype("float32")
df_opt["qty"]    = df_opt["qty"].astype("int8")     # qty mieści się w 0-127
df_opt["is_bad"] = df_opt["is_bad"].astype("int8")

# category dtype: deduplikuje stringi (słownik unikalnych wartości)
df_opt["category"] = df_opt["category"].astype("category")
df_opt["city"]     = df_opt["city"].astype("category")

before = df.memory_usage(deep=True).sum()
after  = df_opt.memory_usage(deep=True).sum()

print(df_opt.dtypes)
print()
print(f"Przed: {before/1024:.2f} KB")
print(f"Po:    {after/1024:.2f} KB")
print(f"Redukcja: {(1 - after/before)*100:.0f}%")

**Kiedy float16?**
- Modele ML (XGBoost, PyTorch) akceptują float32: float16 bywa za mało precyzyjne
- Do przechowywania i EDA: float32 jest bezpiecznym kompromisem
- `category` = największy zysk dla kolumn z małą liczbą unikalnych wartości

## 2. datetime: praca z datami

Po sparsowaniu do `datetime64` dostajemy `.dt` accessor z dziesiątkami atrybutów.

In [ ]:
# Parsowanie: pandas sam rozpoznaje większość formatów
df_opt["purchase_date"] = pd.to_datetime(df["purchase_date"])

print(df_opt["purchase_date"].dtype)
print(df_opt["purchase_date"].head(3))

In [ ]:
dt = df_opt["purchase_date"].dt   # accessor

print(df_opt.assign(
    rok        = dt.year,
    miesiac    = dt.month,
    dzien_tyg  = dt.day_name(),      # Monday, Tuesday...
    godzina    = dt.hour,
    weekend    = dt.dayofweek.isin([5, 6]).astype(int),
)[["order_id","purchase_date","rok","miesiac","dzien_tyg","godzina","weekend"]].to_string())

In [ ]:
# Różnica dat: liczba dni między zdarzeniami
reference_date = pd.Timestamp("2021-06-30")
df_opt["days_since_purchase"] = (reference_date - df_opt["purchase_date"]).dt.days

print(df_opt[["order_id","purchase_date","days_since_purchase"]].to_string())

## 3. pandas StringDtype: zamiast `object`

Domyślnie Pandas przechowuje tekst jako `object` (Python str + wskaźniki).
`pd.StringDtype()` to dedykowany typ dla tekstu, lepsze NA, czytelniejszy kod.

In [ ]:
s_object = pd.Series(["Buty Nike", "Kurtka Zara", None, "Buty Adidas"], dtype="object")
s_string = pd.Series(["Buty Nike", "Kurtka Zara", None, "Buty Adidas"], dtype="string")

print("object:", s_object.dtype, "| NA:", s_object[2])
print("string:", s_string.dtype, "| NA:", s_string[2])
print()
# object zwraca None (Python), string zwraca pd.NA (Pandas-native)
print("type(NA) object:", type(s_object[2]))
print("type(NA) string:", type(s_string[2]))

## `.str` accessor: operacje na tekście

In [ ]:
products = df_opt["category"].astype("string")
cities   = df_opt["city"].astype("string")

# Zawiera podciąg
print("Zawiera 'shoes':")
print(df_opt["category"].astype("string").str.contains("shoes").tolist())
print()

# Zamiana
print("Zamiana 'shoes' → 'buty':")
print(df_opt["category"].astype("string").str.replace("shoes", "buty").tolist())

In [ ]:
# Wyciąganie wzorca (regex): np. pierwsza litera jako prefix
name_col = pd.Series(["Anna Kowalska", "Jan Wiśniewski", "Maria Nowak"], dtype="string")

print("upper:  ", name_col.str.upper().tolist())
print("split:  ", name_col.str.split(" ").tolist())
print("imie:   ", name_col.str.split(" ").str[0].tolist())
print("inicjał:", name_col.str.extract(r"^(\w)")[0].tolist())
print("dlugosc:", name_col.str.len().tolist())

In [ ]:
# Typ string vs object w operacjach zwracających bool
s = pd.Series(["shoes", None, "clothes"], dtype="string")
result = s.str.startswith("s")

print(result)
print()
print("Typ wyniku:", result.dtype)
print("NA zostaje NA, nie False jak w object dtype")